In [ ]:
import cv2
import os
import numpy as np
from skimage import io
from glob import glob
from tqdm import tqdm
import histomicstk as htk

from histomicstk.preprocessing.color_normalization import reinhard
from histomicstk.preprocessing.color_conversion import rgb_to_lab
import skimage.io
import matplotlib.pyplot as plt
from collections import defaultdict
import re
import shutil

Reinhard Normalization Checker

In [ ]:

img = skimage.io.imread("dataset/training_data/train/Images/3.png")
ref = skimage.io.imread("dataset/color_refrence.png")
img_lab = rgb_to_lab(img)
ref_lab = rgb_to_lab(ref)
img_means = np.mean(img_lab.reshape(-1, 3), axis=0)
img_stds  = np.std(img_lab.reshape(-1, 3), axis=0)

ref_means = np.mean(ref_lab.reshape(-1, 3), axis=0)
ref_stds  = np.std(ref_lab.reshape(-1, 3), axis=0)
print("Image LAB mean/std:", img_means, img_stds)
print("Ref   LAB mean/std:", ref_means, ref_stds)
mean_diff = np.abs(img_means - ref_means)
std_diff  = np.abs(img_stds - ref_stds)

print("Mean difference:", mean_diff)
print("Std difference:", std_diff)
threshold = 1.0
if np.all(mean_diff < threshold) and np.all(std_diff < threshold):
    print("Image appears to be Reinhard normalized to reference.")
else:
    print("Image does not appear to be Reinhard normalized.")


In [ ]:
from PIL import Image

for path in ["dataset/training_data/valid/Images/5.png", "dataset/color_reference.png","dataset/Color_Normalized/valid/4.png"]:
    im = Image.open(path)
    print(path, "mode:", im.mode)

In [ ]:
ref_path = "dataset/color_reference.png"        
src_folder      = "dataset/training_data/valid/Images"              
dest_folder     = "dataset/Color_Normalized/"         
os.makedirs(dest_folder, exist_ok=True)


Count Files in the Dataset Directory

In [ ]:

def count_images_in_directory(folder_path, extensions=('*.png')):
    total = 0
    for ext in extensions:
        image_paths = glob.glob(os.path.join(folder_path, ext))
        total += len(image_paths)
    return total
folder = "dataset/training_data/color_normlaized/train/Images"
num_images = count_images_in_directory(folder)
print(f"Total number of images in '{folder}': {num_images}")

Convert the Images to 1204*768 and zero padded

In [ ]:

def resize_with_padding(image, target_size=(1024, 768)):
    target_w, target_h = target_size
    h, w = image.shape[:2]

    scale = min(target_w / w, target_h / h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)

    padded = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2
    padded[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized

    return padded

def process_directory(input_dir, output_dir, target_size=(1024, 768)):
    os.makedirs(output_dir, exist_ok=True)
    image_paths = glob(os.path.join(input_dir, "*.JPG"))  

    for path in image_paths:
        img = cv2.imread(path)
        if img is None:
            print(f"Skipping unreadable file: {path}")
            continue

        padded_img = resize_with_padding(img, target_size)

        base_filename = os.path.splitext(os.path.basename(path))[0]
        output_path = os.path.join(output_dir, base_filename + ".png")
        cv2.imwrite(output_path, padded_img)
        print(f"Saved: {output_path}")

# Example usage
if __name__ == "__main__":
    input_directory = "dataset/Other_training/raw_images/"      
    output_directory = "dataset/Other_training/nm_images/"   
    process_directory(input_directory, output_directory)




Identify duplicate filenames in the dataset

In [ ]:
root_dir = "dataset/training_data/hough_transform/"
image_extensions = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
image_locations = defaultdict(list)
for dirpath, dirnames, filenames in os.walk(root_dir):
    for file in filenames:
        if file.lower().endswith(image_extensions):
            image_locations[file].append(dirpath)
print("Duplicate filenames found in different directories:\n")
for filename, paths in image_locations.items():
    if len(paths) > 1:
        print(f"{filename} found in:")
        for path in paths:
            print(f"└── {path}")
        print()

Overlay annotation on the images to check the proper annotation .txt file is used or not

In [ ]:

def annotate_image_with_polygons(image_path, label_path):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    with open(label_path, 'r') as f:
        for line in f:
            parts    = line.strip().split()
            class_id = int(parts[0])
            coords   = list(map(float, parts[1:]))
            pts = []
            for i in range(0, len(coords), 2):
                x = int(coords[i]   * w)
                y = int(coords[i+1] * h)
                pts.append((x, y))

            poly_color = (0, 200, 0) if class_id == 0 else (200, 0, 0)
            pts_arr = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
            cv2.polylines(img, [pts_arr], isClosed=True, color=poly_color, thickness=2)
            cv2.putText(
                img,
                f"Class {class_id}",
                (pts[0][0], pts[0][1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                poly_color,
                1,
                cv2.LINE_AA
            )
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
image_paths = [
    "dataset/GeneST_final_data/train/images/3_png.rf.74d3d9d7bf303980a054845e086cb88d.jpg",
    "dataset/GeneST_final_data/train/images/50HD0002_2min4704_png.rf.06d932ebe98cdaa3b7a7bdc2ce0f579f.jpg",
    "dataset/GeneST_final_data/train/images/multi4_png.rf.ca75268575f94f2d2c3bedc9b3417ec3.jpg",
    "dataset/GeneST_final_data/train/images/multi6_png.rf.b8f3258c88534b3c16907c8d233ac497.jpg"
]
label_paths = [
    "dataset/GeneST_final_data/train/labels/3_png.rf.74d3d9d7bf303980a054845e086cb88d.txt",
    "dataset/GeneST_final_data/train/labels/50HD0002_2min4704_png.rf.06d932ebe98cdaa3b7a7bdc2ce0f579f.txt",
    "dataset/GeneST_final_data/train/labels/multi4_png.rf.ca75268575f94f2d2c3bedc9b3417ec3.txt",
    "dataset/GeneST_final_data/train/labels/multi6_png.rf.b8f3258c88534b3c16907c8d233ac497.txt"
]


# assert len(image_paths) == len(label_paths) == 4, "Provide exactly 4 pairs."

annotated_images = [
    annotate_image_with_polygons(img, lbl)
    for img, lbl in zip(image_paths, label_paths)
]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, img, path in zip(axes.ravel(), annotated_images, image_paths):
    ax.imshow(img)
    ax.set_title(os.path.basename(path))
    ax.axis("off")

plt.tight_layout()
output_file = "annotated_polygons_only_grid.png"
fig.savefig(output_file, dpi=600, bbox_inches='tight')
print(f"Saved: {output_file}")

plt.show()


In [ ]:
roboflow_images_dir = "dataset/GeneST_final_data/train/images"
roboflow_labels_dir = "dataset/GeneST_final_data/train/labels"
image_basenames = defaultdict(list)
label_basenames = defaultdict(list)

for file in os.listdir(roboflow_images_dir):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        match = re.match(r"^(.*?)\.rf\..*?\.(jpg|jpeg|png)$", file, re.IGNORECASE)
        if match:
            base = match.group(1).replace("_", ".")  
            image_basenames[base].append(file)


for file in os.listdir(roboflow_labels_dir):
    if file.lower().endswith(".txt"):
        match = re.match(r"^(.*?)\.rf\..*?\.txt$", file, re.IGNORECASE)
        if match:
            base = match.group(1).replace("_", ".")  
            base = os.path.splitext(base)[0] + ".txt"
            label_basenames[base].append(file)

print("Duplicate image basenames:")
for base, files in image_basenames.items():
    if len(files) > 1:
        print(f"\n {base} → {len(files)} duplicates:")
        for f in files:
            print(f"   └─ {f}")

print("\nDuplicate label basenames:")
for base, files in label_basenames.items():
    if len(files) > 1:
        print(f"\n {base} → {len(files)} duplicates:")
        for f in files:
            print(f"   └─ {f}")


  # Duplicate file names found and renamed:
  # 50HD0054.png → 50HD0054_from_ht_images.png `
  # 50HD0058.png → 50HD0058_from_ht_images.png `
  # 50HD0070.png → 50HD0070_from_ht_images.png
  # 50HD0066.png → 50HD0066_from_ht_images.png `
  # 50HD0059.png → 50HD0059_from_ht_images.png `
  # 50HD0068.png → 50HD0068_from_ht_images.png `
  # 50HD0055.png → 50HD0055_from_ht_images.png `
  # 50HD0063.png → 50HD0063_from_ht_images.png `
  # 50HD0065.png → 50HD0065_from_ht_images.png 
  # 50HD0057.png → 50HD0057_from_ht_images.png `
  # 50HD0060.png → 50HD0060_from_ht_images.png `
  # 50HD0062.png → 50HD0062_from_ht_images.png `
  # 50HD0053.png → 50HD0053_from_ht_images.png `
  # 50HD0067.png → 50HD0067_from_ht_images.png `
  # 50HD0056.png → 50HD0056_from_ht_images.png `
  # 50HD0064.png → 50HD0064_from_ht_images.png `
  # 50HD0069.png → 50HD0069_from_ht_images.png
  # 50HD0061.png → 50HD0061_from_ht_images.png `
 

In [ ]:

input_dirs = [
    "dataset/training_data/hough_transform/train",
    "dataset/training_data/hough_transform/valid",
    "dataset/training_data/hough_transform/test",
    "dataset/Other_training/ht_images"
]

output_dir = "dataset/temp"
os.makedirs(output_dir, exist_ok=True)
seen_files = set()
duplicates = []

for dir_path in input_dirs:
    dir_name = os.path.basename(os.path.normpath(dir_path))  

    for file in os.listdir(dir_path):
        if file.lower().endswith(".png"):
            src_file = os.path.join(dir_path, file)
            base_name = os.path.basename(file)

            dest_file = os.path.join(output_dir, base_name)
            if base_name in seen_files:
                name_part = os.path.splitext(base_name)[0]
                new_name = f"{name_part}_from_{dir_name}.png"
                dest_file = os.path.join(output_dir, new_name)
                duplicates.append((base_name, new_name))
            else:
                seen_files.add(base_name)
            shutil.copy2(src_file, dest_file)
if duplicates:
    print("\nDuplicate file names found and renamed:")
    for original, new in duplicates:
        print(f"  {original} → {new}")
else:
    print("No duplicate filenames found.")


In [ ]:
image_dir = "dataset/GeneST_final_data/train/images"
lab_dir = "dataset/GeneST_final_data/train/labels"
valid_extensions = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff",".txt")
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(valid_extensions)]
lable_files = [f for f in os.listdir(lab_dir) if f.lower().endswith(valid_extensions)]
print(f"Total number of images: {len(image_files)}")
print(f"Total number of images: {len(lable_files)}")

In [ ]:
image_dir = "dataset/temp_new/images"
lab_dir = "dataset/temp_new/labels"
valid_extensions = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff",".txt")
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(valid_extensions)]
lable_files = [f for f in os.listdir(lab_dir) if f.lower().endswith(valid_extensions)]
print(f"Total number of images: {len(image_files)}")
print(f"Total number of images: {len(lable_files)}")